# 1.

In [8]:
import os
import math
import torch
import torch.nn as nn
from tokenizers import Tokenizer                        #分词
from torchtext.vocab import build_vocab_from_iterator   #构建词袋
from torch.utils.data import Dataset,DataLoader         #数据集与批次数据集
from torch.nn.functional import pad,log_softmax         #pad补边与对齐,log_softmax(转换为概率)
from torch.utils.data import Dataset

#GPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")   #标准的torch.device对象

In [2]:
tokenizer= Tokenizer.from_file("./tokenizer.json")
def en_tokenizer(line):
    """
    """
    tokens = tokenizer.encode(line,add_special_tokens=False).tokens
    return tokens

In [3]:
zh_filepath = "./datasets/train.zh"
def zh_tokenizer(line):
    return list(line.strip().replace("","")) 

In [4]:
zh_vocab = torch.load("vocab_zh.pt",weights_only=False)         #加载对象，存储的对象必须支持序列与反序列
en_vocab = torch.load("vocab_en.pt",weights_only=False)

zh_filepath = "./datasets/train.zh"
en_filepath = "./datasets/train.en"

In [5]:
class TranslateDataset(Dataset):
    def __init__(self):
        #初始化
        self.zh_tokens=self._load_tokens(zh_filepath,zh_tokenizer,zh_vocab)
        self.en_tokens=self._load_tokens(en_filepath,en_tokenizer,en_vocab)

        self.len_tokens = len(self.zh_tokens)

    def __getitem__(self, idx):
        #根据索引返回数据
        return self.en_tokens[idx],self.zh_tokens[idx]

    def __len__(self):
        # 返回数据集长度
        return self.len_tokens

    # def _load_tokens(self,filepath,tokenizer,vocab):
    #     tokens_list = []    #存放 向量化的词
    #     with open(filepath,encoding="utf-8") as fd:
    #         for line in fd:
    #             tokens = tokenizer(line)    #分词
    #             #把token 转为编号（向量化）
    #             num_tokens = [vocab[token] for token in tokens]
    #             #存储到列表
    #             tokens_list.append(num_tokens)
    #     return tokens_list

    def _load_tokens(self, filepath, tokenizer, vocab):
        tokens_list = []
        with open(filepath, encoding="utf-8") as fd:
            for line in fd:
                line = line.strip()
                if not line:
                    continue
                    
                tokens = tokenizer(line)
                num_tokens = [vocab.stoi[token] if token in vocab.stoi else vocab.default_index for token in tokens]
                tokens_list.append(num_tokens)
                
        return tokens_list
                


In [6]:
#测试

ds = TranslateDataset()
print(ds[0])

([9, 2728, 10, 552, 17, 17208, 18075, 25, 3076, 201, 55, 100, 18830, 3651], [10, 38, 1172, 1083, 3168, 163, 692, 396, 83, 99, 12, 3, 1217, 2396, 534, 66])


# 3. 批次数据集

- pytorch:[N,V,...]
    - 对齐

In [14]:
max_length = 72     #每个样本超过72则截断，不足则补齐
def collate_fn(batch):  #batch使用列表的方式，返回样本与标签
    src_list = []
    dst_list = []

    for (src,dst) in batch:
        # 把src 与 dst 转换为张量
        process_src = torch.tensor(src,dtype=torch.int64)
        process_dst = torch.tensor(dst,dtype=torch.int64)

        #对齐
        process_src = pad(process_src, (0, max_length - len(process_src)), mode='constant', value=en_vocab["<pad>"])
        process_dst = pad(process_dst, (0, max_length - len(process_dst)), mode='constant', value=zh_vocab["<pad>"])

        #存储
        src_list.append(process_src)
        dst_list.append(process_dst)

    #转换
    re_src = torch.stack(src_list)
    re_dst = torch.stack(dst_list)

    #目标单独处理，去掉第一个与最后一个
    re_dst_y = re_dst[:,  1:]        #去掉第一个
    re_dst   = re_dst[:, :-1]        #去掉最后一个

    #计算token数目
    n_tokens = (re_dst != zh_vocab["<pad>"]).sum()
    return re_src,re_dst,re_dst_y,n_tokens   #n_tokens对训练没有意义

In [15]:
loader = DataLoader(ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
for src , dst, dst_y , n_tokens in loader:
    print(src.shape)
    print(dst.shape)
    print(dst_y.shape)
    print(n_tokens)
    break;

torch.Size([64, 72])
torch.Size([64, 71])
torch.Size([64, 71])
tensor(994)


# 4. transformer 的模型

## 4.1 位置编码

In [20]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        # 初始化Shape为(max_len, d_model)的PE (positional encoding)
        pe = torch.zeros(max_len, d_model).to(device)
        # 初始化一个tensor [[0, 1, 2, 3, ...]]
        position = torch.arange(0, max_len).unsqueeze(1)
        # 这里就是sin和cos括号中的内容，通过e和ln进行了变换
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        # 计算PE(pos, 2i)
        pe[:, 0::2] = torch.sin(position * div_term)
        # 计算PE(pos, 2i+1)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 为了方便计算，在最外面在unsqueeze出一个batch
        pe = pe.unsqueeze(0)
        # 如果一个参数不参与梯度下降，但又希望保存model的时候将其保存下来
        # 这个时候就可以用register_buffer
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x 为embedding后的inputs，例如(1,7, 128)，batch size为1,7个单词，单词维度为128
        """
        # 将x和positional encoding相加。
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)

# 4.2 翻译模型（Transformer模型）

In [23]:
class TranslateModel(nn.Module):
    def __init__(self,d_model,src_vocab,target_vocab,dropout=0.1):
        #定义模型层
        super(TranslateModel,self).__init__()

        #样本的词嵌入
        self.src_embedding = nn.Embedding(len(src_vocab), d_model, padding_idx=src_vocab["<pad>"])

        #标签的词嵌入
        self.tgt_embedding = nn.Embedding(len(target_vocab), d_model, padding_idx=target_vocab["<pad>"])

        #位置编码
        self.positional_encoding = PositionalEncoding(d_model, dropout, max_len=max_length)     #max_length就是句子的长度

        #构建transform
        self.transformer = nn.Transformer(
            d_model, 
            dropout=dropout,        #输入的数据，第一维必须是批次维度
            batch_first=True,       
            nhead=8,                #多头的数量
            num_encoder_layers=2,   #编码器层数
            num_decoder_layers=2,   #解码器层数
            dim_feedforward=128)    #网络深度

        #预测器
        self.predictor = nn.Linear(d_model, len(target_vocab))


    def forward(self,src,tgt):
        """
        进行前向传递，输出为Decoder的输出。注意，这里并没有使用self.predictor进行预测，
        因为训练和推理行为不太一样，所以放在了模型外面。
        :param src: 原batch后的句子，例如[[0, 12, 34, .., 1, 2, 2, ...], ...]
        :param tgt: 目标batch后的句子，例如[[0, 74, 56, .., 1, 2, 2, ...], ...]
        :return: Transformer的输出，或者说是TransformerDecoder的输出。
        """

        """
        生成tgt_mask，即阶梯型的mask，例如：
        [[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]]
        tgt.size()[-1]为目标句子的长度。
        """
        # print(src.shape, tgt.shape)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size()[-1]).to(device)
        # 掩盖住原句子中<pad>的部分，例如[[False,False,False,..., True,True,...], ...]
        src_key_padding_mask = TranslateModel.get_key_padding_mask(src)
        # 掩盖住目标句子中<pad>的部分
        tgt_key_padding_mask = TranslateModel.get_key_padding_mask(tgt)
        # print(tgt_mask.shape, src_key_padding_mask.shape, tgt_key_padding_mask.shape)
        # print(tgt_mask.dtype, src_key_padding_mask.dtype, tgt_key_padding_mask.dtype)
        # 对src和tgt进行编码
        src = self.src_embedding(src)
        tgt = self.tgt_embedding(tgt)
        # 给src和tgt的token增加位置信息
        src = self.positional_encoding(src)
        tgt = self.positional_encoding(tgt)
        
        # 将准备好的数据送给transformer
        out = self.transformer(src, tgt,
                               tgt_mask=tgt_mask,
                               src_key_padding_mask=src_key_padding_mask,
                               tgt_key_padding_mask=tgt_key_padding_mask)

        """
        这里直接返回transformer的结果。因为训练和推理时的行为不一样，
        所以在该模型外再进行线性层的预测。
        """
        return out

    @staticmethod
    def get_key_padding_mask(tokens):
        return tokens == zh_vocab["<pad>"]      #返回一个逻辑值（判定）
    

- 测试

In [24]:
model = TranslateModel(200, en_vocab, zh_vocab) #d_model与多头的数量是倍数
model = model.to(device)

for src , tgt,tgt_y,n_tokens in loader:
    src = src.to(device)
    tgt = tgt.to(device)
    y = model(src,tgt)
    print(y.shape)
    break

torch.Size([64, 71, 200])


/Users/logicye/Code/ai_learning/projects/13_transformer_structured_text/.venv/lib/python3.13/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


# 5. 训练

In [27]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

- 损失函数

In [29]:
class TranslationLoss(nn.Module):

    def __init__(self):
        super(TranslationLoss, self).__init__()
        # 使用KLDivLoss，不需要知道里面的具体细节。
        self.criterion = nn.KLDivLoss(reduction="sum")
        self.padding_idx = zh_vocab["<pad>"]

    def forward(self, x, target):
        """
        损失函数的前向传递
        :param x: 将Decoder的输出再经过predictor线性层之后的输出。
                  也就是Linear后、Softmax前的状态
        :param target: tgt_y。也就是label，例如[[1, 34, 15, ...], ...]
        :return: loss
        """

        """
        由于KLDivLoss的input需要对softmax做log，所以使用log_softmax。
        等价于：log(softmax(x))
        """
        x = log_softmax(x, dim=-1)

        """
        构造Label的分布，也就是将[[1, 34, 15, ...]] 转化为:
        [[[0, 1, 0, ..., 0],
          [0, ..., 1, ..,0],
          ...]],
        ...]
        """
        # 首先按照x的Shape构造出一个全是0的Tensor
        true_dist = torch.zeros(x.size()).to(device)
        # 将对应index的部分填充为1
        true_dist.scatter_(1, target.data.unsqueeze(1), 1)
        # 找出<pad>部分，对于<pad>标签，全部填充为0，没有1，避免其参与损失计算。
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)

        # 计算损失
        return self.criterion(x, true_dist.clone().detach())

In [30]:
criteria = TranslationLoss()

In [ ]:
epochs = 10
step = 0
save_after_step = 5000
model.train()
for epoch in range(epochs):
    for index, data in enumerate(loader):
        # 生成数据
        src, tgt, tgt_y, n_tokens = data
        src, tgt, tgt_y = src.to(device), tgt.to(device), tgt_y.to(device)
        # 清空梯度
        optimizer.zero_grad()
        # 进行transformer的计算
        out = model(src, tgt)
        # 将结果送给最后的线性层进行预测
        out = model.predictor(out)
        """
        计算损失。由于训练时我们的是对所有的输出都进行预测，所以需要对out进行reshape一下。
                我们的out的Shape为(batch_size, 词数, 词典大小)，view之后变为：
                (batch_size*词数, 词典大小)。
                而在这些预测结果中，我们只需要对非<pad>部分进行，所以需要进行正则化。也就是
                除以n_tokens。
        """
        loss = criteria(out.contiguous().view(-1, out.size(-1)), tgt_y.contiguous().view(-1)) / n_tokens
        # 计算梯度
        loss.backward()
        # 更新参数
        optimizer.step()
        step += 1
        if step != 0 and step % save_after_step == 0:
            print(F"论数：{epoch},\t损失值：{loss.detach().item()}")        
            torch.save(model, F"datasets/model_{step:02d}.pt")

KeyboardInterrupt: 